In [ ]:
import pandas as pd
from datetime import timedelta

df = pd.read_csv("olist_order_summary.csv")
df['order_date'] = pd.to_datetime(df['order_date'])

snapshot_date = df['order_date'].max() + timedelta(days=1)

rfm = df.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'count',
    'payment_value': 'sum'
}).reset_index()

rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary']

# RFM scoring (1–5 scale)
rfm['R'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1])
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5])
rfm['rfm_score'] = rfm['R'].astype(int) + rfm['F'].astype(int) + rfm['M'].astype(int)

rfm.to_csv("olist_customer_rfm.csv", index=False)
